# Unknown Counts Per Unique Airport Code

For each unique airport code, count Unknown values in FromCity and ToCity (vectorized).

In [1]:
import pandas as pd
import numpy as np

# Load data
print("Loading data...")
mapping_df = pd.read_excel('gs://agntworks-data-dev/wheelsup/raw/Mappings/ICAO to City Mapping.xlsx')
revenue_df = pd.read_parquet('gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet')

# Get unique codes and mapped codes
unique_codes = set(revenue_df['FromAirport'].dropna().unique()) | set(revenue_df['ToAirport'].dropna().unique())
mapped_codes = set(mapping_df['displayCode'].dropna().unique())

print(f"Total unique airport codes: {len(unique_codes)}")
print(f"Codes in mapping: {len(unique_codes & mapped_codes)}")
print(f"Codes missing from mapping: {len(unique_codes - mapped_codes)}")

Loading data...
Total unique airport codes: 2631
Codes in mapping: 2010
Codes missing from mapping: 621


In [2]:
# Vectorized: Count FromCity Unknown per FromAirport code
print("\nComputing unknown counts (vectorized)...")

from_city_is_unknown = revenue_df['FromCity'] == 'Unknown'
from_unknown_counts = revenue_df[from_city_is_unknown].groupby('FromAirport').size()

# Vectorized: Count ToCity Unknown per ToAirport code
to_city_is_unknown = revenue_df['ToCity'] == 'Unknown'
to_unknown_counts = revenue_df[to_city_is_unknown].groupby('ToAirport').size()

print("✓ Done")


Computing unknown counts (vectorized)...
✓ Done


In [3]:
# Build result with all codes
results = []
for code in sorted(unique_codes):
    from_unknown = from_unknown_counts.get(code, 0)
    to_unknown = to_unknown_counts.get(code, 0)
    total_unknown = from_unknown + to_unknown
    status = "✓ Mapped" if code in mapped_codes else "✗ MISSING"

    results.append({
        'Code': code,
        'FromCity Unknown': from_unknown,
        'ToCity Unknown': to_unknown,
        'Total Unknown': total_unknown,
        'Status': status
    })

df = pd.DataFrame(results)
df = df.sort_values('Total Unknown', ascending=False)

print("\n" + "="*90)
print("UNKNOWN COUNTS PER UNIQUE AIRPORT CODE")
print("="*90)
print(f"\n{df.to_string(index=False)}")


UNKNOWN COUNTS PER UNIQUE AIRPORT CODE

Code  FromCity Unknown  ToCity Unknown  Total Unknown    Status
KT82              1667            1727           3394 ✗ MISSING
K5B2               746             767           1513 ✗ MISSING
K27K               720             724           1444 ✗ MISSING
K0A9               718             726           1444 ✗ MISSING
KF45               611             594           1205 ✗ MISSING
K29D               590             605           1195 ✗ MISSING
K1R8               557             533           1090 ✗ MISSING
K1A5               441             444            885 ✗ MISSING
KF82               456             421            877 ✗ MISSING
K5C1               421             427            848 ✗ MISSING
KX60               379             452            831 ✗ MISSING
K11R               387             397            784 ✗ MISSING
K8A0               351             318            669 ✗ MISSING
K46U               306             320            626 ✗ MISSING

In [4]:
# Summary statistics
print("\n" + "="*90)
print("SUMMARY")
print("="*90)

print(f"\nTotal unique airport codes: {len(df)}")
print(f"Codes with Unknown cities: {(df['Total Unknown'] > 0).sum()}")
print(f"Codes with 0 Unknown cities: {(df['Total Unknown'] == 0).sum()}")

print(f"\nUnknown Counts:")
print(f"  FromCity Unknown total: {df['FromCity Unknown'].sum()}")
print(f"  ToCity Unknown total: {df['ToCity Unknown'].sum()}")
print(f"  Grand Total Unknown: {df['Total Unknown'].sum()}")


SUMMARY

Total unique airport codes: 2631
Codes with Unknown cities: 621
Codes with 0 Unknown cities: 2010

Unknown Counts:
  FromCity Unknown total: 20514
  ToCity Unknown total: 20622
  Grand Total Unknown: 41136


In [5]:
# Breakdown by mapping status
print("\n" + "="*90)
print("BREAKDOWN BY MAPPING STATUS")
print("="*90)

print(f"\nMissing from Mapping:")
missing_df = df[df['Status'] == '✗ MISSING']
print(f"  Total missing codes: {len(missing_df)}")
print(f"  Missing codes with Unknown: {(missing_df['Total Unknown'] > 0).sum()}")
print(f"  Unknown count in missing codes: {missing_df['Total Unknown'].sum()}")

print(f"\nMapped in Mapping:")
mapped_df = df[df['Status'] == '✓ Mapped']
print(f"  Total mapped codes: {len(mapped_df)}")
print(f"  Mapped codes with Unknown: {(mapped_df['Total Unknown'] > 0).sum()}")
print(f"  Unknown count in mapped codes: {mapped_df['Total Unknown'].sum()}")


BREAKDOWN BY MAPPING STATUS

Missing from Mapping:
  Total missing codes: 621
  Missing codes with Unknown: 621
  Unknown count in missing codes: 41136

Mapped in Mapping:
  Total mapped codes: 2010
  Mapped codes with Unknown: 0
  Unknown count in mapped codes: 0


In [6]:
# Show missing codes with Unknown
print("\n" + "="*90)
print("MISSING CODES WITH UNKNOWN CITIES")
print("="*90)

missing_with_unknown = missing_df[missing_df['Total Unknown'] > 0].sort_values('Total Unknown', ascending=False)
print(f"\n{missing_with_unknown.to_string(index=False)}")


MISSING CODES WITH UNKNOWN CITIES

Code  FromCity Unknown  ToCity Unknown  Total Unknown    Status
KT82              1667            1727           3394 ✗ MISSING
K5B2               746             767           1513 ✗ MISSING
K27K               720             724           1444 ✗ MISSING
K0A9               718             726           1444 ✗ MISSING
KF45               611             594           1205 ✗ MISSING
K29D               590             605           1195 ✗ MISSING
K1R8               557             533           1090 ✗ MISSING
K1A5               441             444            885 ✗ MISSING
KF82               456             421            877 ✗ MISSING
K5C1               421             427            848 ✗ MISSING
KX60               379             452            831 ✗ MISSING
K11R               387             397            784 ✗ MISSING
K8A0               351             318            669 ✗ MISSING
K46U               306             320            626 ✗ MISSING
KM19

In [7]:
# Export to CSV
df.to_csv('unknown_counts_per_code.csv', index=False)
print("\n✓ Exported to: unknown_counts_per_code.csv")


✓ Exported to: unknown_counts_per_code.csv


In [11]:
len(missing_with_unknown[missing_with_unknown["Status"]=="✗ MISSING"])

621